# Mini Project: Sentiment Assistant with BERT Fine-Tuning

**Goal:** fine-tune `bert-base-uncased` on the IMDB movie-review dataset so that it can be used as a *sentiment assistant* (e.g. to triage customer feedback for a support team).

Plan:
1. Setup & hardware check
2. Load and inspect IMDB
3. Tokenise with `BertTokenizer` (padding, truncation, attention masks)
4. Fine-tune `BertForSequenceClassification` with the Hugging Face `Trainer`
5. Evaluate on held-out test reviews (accuracy, F1, confusion matrix, error analysis)
6. Build a `predict_sentiment` helper and test it on support-style messages
7. Reflection: acceptable error rates, levers, guardrails, stakeholders

> **Compute note.** This notebook was executed on a CPU-only machine (4 cores, no GPU), where one training step of BERT-base at 256 tokens with batch 16 takes ≈13 s — the full 25 000-review training set for 2 epochs would take ~11 h.
> The code therefore uses a **random, class-balanced subset** (`N_TRAIN` / `N_VAL` / `N_TEST` below) when no GPU is detected. On Colab with a GPU (Runtime → Change runtime type → T4), the same notebook automatically uses the **full** dataset.

## 1. Setup

In [ ]:
!pip install -q torch "transformers<5" datasets accelerate evaluate scikit-learn

In [ ]:
import os
os.environ["USE_TF"] = "0"              # avoid TensorFlow/Keras imports inside transformers
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import evaluate
from datasets import load_dataset
from transformers import (BertTokenizer, BertForSequenceClassification,
                          TrainingArguments, Trainer, set_seed)
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import transformers, datasets

SEED = 42
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("Device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else f"CPU threads: {torch.get_num_threads()}")

## 2. Load the IMDB dataset

In [ ]:
dataset = load_dataset("imdb")
dataset

In [ ]:
# Peek at a couple of shuffled samples
for ex in dataset["train"].shuffle(seed=SEED).select(range(2)):
    print("label:", "Positive" if ex["label"] == 1 else "Negative")
    print(ex["text"][:400], "...\n")

In [ ]:
# Class balance and review length
train_df = dataset["train"].to_pandas()
print(train_df["label"].value_counts().rename({0: "neg", 1: "pos"}))
lengths = train_df["text"].str.split().str.len()
print(lengths.describe().round(0))

plt.figure(figsize=(7, 3))
plt.hist(lengths.clip(upper=1200), bins=60, color="steelblue")
plt.axvline(200, color="red", ls="--", label="≈256 word-pieces")
plt.xlabel("words per review"); plt.ylabel("count"); plt.title("IMDB review length"); plt.legend()
plt.show()
print(f"Share of reviews longer than ~200 words (will be truncated): {(lengths > 200).mean():.1%}")

### Subset selection (CPU only)
On GPU we keep the full splits. On CPU we draw a balanced random subset of the training set and carve a small **validation** set out of it (used for per-epoch evaluation), and keep a separate balanced subset of the official **test** split for the final, untouched evaluation.

In [ ]:
USE_FULL = device.type == "cuda"
N_TRAIN, N_VAL, N_TEST = 2000, 500, 2000     # used only on CPU

def balanced_subset(split, n, seed):
    split = split.shuffle(seed=seed)
    neg = split.filter(lambda x: x["label"] == 0).select(range(n // 2))
    pos = split.filter(lambda x: x["label"] == 1).select(range(n // 2))
    return datasets.concatenate_datasets([neg, pos]).shuffle(seed=seed)

if USE_FULL:
    split = dataset["train"].train_test_split(test_size=0.1, seed=SEED, stratify_by_column="label")
    raw_train, raw_val, raw_test = split["train"], split["test"], dataset["test"]
else:
    pool = balanced_subset(dataset["train"], N_TRAIN + N_VAL, SEED)
    split = pool.train_test_split(test_size=N_VAL, seed=SEED, stratify_by_column="label")
    raw_train, raw_val = split["train"], split["test"]
    raw_test = balanced_subset(dataset["test"], N_TEST, SEED)

for name, d in [("train", raw_train), ("val", raw_val), ("test", raw_test)]:
    print(f"{name:5s}: {len(d):6d} reviews | positive share = {np.mean(d['label']):.2f}")

## 3. Tokenisation

In [ ]:
MAX_LENGTH = 256
BATCH_SIZE = 16

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",          # pad every review to MAX_LENGTH
        truncation=True,               # cut longer reviews
        max_length=MAX_LENGTH,
        return_attention_mask=True,    # 1 = real token, 0 = padding
        return_token_type_ids=True,    # single-sentence task -> all zeros
    )

def prepare(ds):
    ds = ds.map(tokenize_batch, batched=True, remove_columns=["text"])
    ds = ds.rename_column("label", "labels")
    ds.set_format("torch", columns=["input_ids", "token_type_ids", "attention_mask", "labels"])
    return ds

train_ds, val_ds, test_ds = prepare(raw_train), prepare(raw_val), prepare(raw_test)
print(train_ds)

sample = train_ds[0]
print("input_ids[:20]     :", sample["input_ids"][:20].tolist())
print("decoded            :", tokenizer.decode(sample["input_ids"][:20]))
print("attention_mask sum :", int(sample["attention_mask"].sum()), "/", MAX_LENGTH)

## 4. Model & Trainer

In [ ]:
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2,
                                                      id2label={0: "Negative", 1: "Positive"},
                                                      label2id={"Negative": 0, "Positive": 1})
model.to(device)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.1f} M")

In [ ]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
            "f1": f1.compute(predictions=preds, references=labels)["f1"]}

In [ ]:
EPOCHS = 2

training_args = TrainingArguments(
    output_dir="./bert-imdb",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="no",               # keep the disk clean; the final model is saved explicitly below
    logging_steps=10 if not USE_FULL else 100,
    report_to="none",
    seed=SEED,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,              # validation set for per-epoch monitoring; the test set stays untouched
    compute_metrics=compute_metrics,
)

In [ ]:
t0 = time.time()
history = trainer.train()
print(f"Training time: {(time.time() - t0)/60:.1f} min")
history

In [ ]:
logs = pd.DataFrame(trainer.state.log_history)
train_logs = logs.dropna(subset=["loss"])[["step", "epoch", "loss"]]
eval_logs = logs.dropna(subset=["eval_loss"])[["epoch", "eval_loss", "eval_accuracy", "eval_f1"]]
display(eval_logs.round(4))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(train_logs["epoch"], train_logs["loss"], label="train loss (per logging step)")
ax[0].plot(eval_logs["epoch"], eval_logs["eval_loss"], "o-", label="validation loss")
ax[0].set_xlabel("epoch"); ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(eval_logs["epoch"], eval_logs["eval_accuracy"], "o-", label="val accuracy")
ax[1].plot(eval_logs["epoch"], eval_logs["eval_f1"], "s--", label="val F1")
ax[1].set_xlabel("epoch"); ax[1].set_title("Validation metrics"); ax[1].legend()
plt.tight_layout(); plt.show()

## 5. Evaluation on the held-out test reviews

In [ ]:
eval_metrics = trainer.evaluate(test_ds, metric_key_prefix="test")
eval_metrics

In [ ]:
pred_out = trainer.predict(test_ds)
logits = pred_out.predictions
y_true = pred_out.label_ids
probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
y_pred = probs.argmax(axis=1)
conf = probs.max(axis=1)

print(classification_report(y_true, y_pred, target_names=["Negative", "Positive"], digits=3))
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=["Negative", "Positive"]).plot(cmap="Blues")
plt.title("Test confusion matrix"); plt.show()

In [ ]:
# How does accuracy depend on model confidence?  (useful to design an "escalate to a human" rule)
res = pd.DataFrame({"text": raw_test["text"], "label": y_true, "pred": y_pred, "conf": conf})
res["correct"] = res["label"] == res["pred"]

rows = []
for thr in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    kept = res[res["conf"] >= thr]
    rows.append({"confidence ≥": thr, "coverage": len(kept) / len(res),
                 "accuracy on kept": kept["correct"].mean(), "error rate on kept": 1 - kept["correct"].mean()})
thr_table = pd.DataFrame(rows)
display(thr_table.round(3))

plt.figure(figsize=(7, 3))
plt.hist([res.loc[res.correct, "conf"], res.loc[~res.correct, "conf"]], bins=20, stacked=True,
         label=["correct", "wrong"], color=["seagreen", "crimson"])
plt.xlabel("model confidence (max softmax prob)"); plt.ylabel("reviews"); plt.legend(); plt.title("Confidence of correct vs wrong predictions")
plt.show()

In [ ]:
# A few of the most confident mistakes
pd.set_option("display.max_colwidth", 300)
worst = res[~res.correct].sort_values("conf", ascending=False).head(5).copy()
worst["label"] = worst["label"].map({0: "Negative", 1: "Positive"})
worst["pred"] = worst["pred"].map({0: "Negative", 1: "Positive"})
worst["text"] = worst["text"].str.slice(0, 300) + "..."
display(worst[["label", "pred", "conf", "text"]].round(3))

### To-Do: what error rate is acceptable for a support team?

ERROR_RATE_PLACEHOLDER

## 6. `predict_sentiment` helper

In [ ]:
def predict_sentiment(text):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=MAX_LENGTH)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0].cpu()
    label = model.config.id2label[int(probs.argmax())]
    return label, float(probs.max())

predict_sentiment("The onboarding emails were confusing, but the agent fixed everything politely.")

In [ ]:
support_messages = [
    "The onboarding emails were confusing, but the agent fixed everything politely.",
    "I have been waiting two weeks for a refund and nobody answers my emails.",
    "Fantastic service, the issue was solved in five minutes. Thank you!",
    "The app crashes every time I open it. Useless.",
    "It's okay I guess, nothing special.",
    "Great, another update that breaks everything. Just great.",
]
for msg in support_messages:
    label, p = predict_sentiment(msg)
    flag = "  -> escalate to human" if p < 0.80 else ""
    print(f"{label:8s} ({p:.3f}){flag} | {msg}")

In [ ]:
# Save the fine-tuned model + tokenizer so the assistant can be reloaded later
trainer.save_model("./bert-imdb/final")
tokenizer.save_pretrained("./bert-imdb/final")
print(os.listdir("./bert-imdb/final"))

## 7. Reflection

REFLECTION_PLACEHOLDER